# Capa Silver — Clustering de Urbanización (CDMX)

%md

**Objetivo del pipeline:** construir, a partir de la capa Bronze (ITER, DENUE x9, Marco Geoestadístico, AGEB, Manzanas),
una tabla Gold a **nivel manzana** con las 4 variables del estudio, lista para K-Means, clustering jerárquico y DBSCAN.

| Variable | Definición | Fuente bronze |
|---|---|---|
| `densidad_urbana` | pob_total (municipio) / superficie_total_km2 (municipio) — densidad bruta, asignada como atributo contextual a cada manzana | `bronze_ITER`, `bronze_marco_geoestadistico` |
| `densidad_urbana_neta` | pob_total (municipio) / superficie_urbana_km2 (mancha urbana real, unión de manzanas) | `bronze_ITER`, `bronze_manzanas` |
| `indice_entropia_negocios` | Entropía de Shannon-Wiener sobre 3 categorías SCIAN activas (Comercio, Servicios, Industria); la categoría "Otros" no se presenta en las fuentes DENUE utilizadas, por lo que la normalización usa N=3 |
| `personas_mayores_o_iguales_18` | Suma de `P_18YMAS` por manzana | `bronze_ITER` |

**Nota metodológica:** `densidad_urbana` y `densidad_urbana_neta` se calculan a nivel **municipal**
(porque `superficie_total_km2` y `superficie_urbana_km2` son magnitudes de área que solo tienen sentido agregadas
geográficamente) y luego se **propagan a cada manzana** como atributo contextual (todas las manzanas de un mismo
municipio comparten el mismo valor de estas dos variables(esto es una limitación MAUP)). Lo que aporta variación intra-municipal real es la
combinación con `personas_mayores_o_iguales_18` e `indice_entropia_negocios`, que sí se calculan por manzana.
Esto es una práctica común en geografía urbana cuantitativa (atributos de contexto + atributos locales combinados
en clustering), pero conviene declararlo explícitamente en la sección de metodología del artículo.

%md
## 1. Silver ITER — Demografía por manzana.

`bronze_ITER` trae columnas demográficas tipadas como **string** (incluyendo `POBTOT` y `P_18YMAS`), y usa elcampo `*9*` (valor convencional de INEGI) para "información reservada por confidencialidad estadística" enmanzanas con muy poca población. Hay que castear a numérico y tratar esos casos como nulos, no como ceros.

In [0]:
# ============ Carga de tablas Bronce ============

# ITER (demografía histórica)
df_iter_2020 = spark.table("bronze_iter_2020")
df_iter_2010 = spark.table("bronze_iter_2010")

# RESAGEBURB (censo a nivel AGEB/manzana)
df_ageburb_2020 = spark.table("bronze_ageburb_2020")
df_ageburb_2010 = spark.table("bronze_ageburb_2010")

# Malla geoestadística
df_municipios = spark.table("bronze_marco_geoestadistico")
df_ageb = spark.table("bronze_ageb")
df_manzanas = spark.table("bronze_manzanas")

# DENUE (economía / entropía de negocios)
df_bancos = spark.table("bronze_bancos")
df_comida = spark.table("bronze_comida")
df_corporativos = spark.table("bronze_corporativos")
df_deportes = spark.table("bronze_deportes")
df_negocios = spark.table("bronze_negocios")
df_negocios_tiendas = spark.table("bronze_negocios_tiendas")
df_restaurantes = spark.table("bronze_restaurantes")
df_tiendas = spark.table("bronze_tiendas")
df_transporte = spark.table("bronze_transporte")

# Verificación rápida de conteos
tablas = {
    "iter_2020": df_iter_2020, "iter_2010": df_iter_2010,
    "ageburb_2020": df_ageburb_2020, "ageburb_2010": df_ageburb_2010,
    "municipios": df_municipios, "ageb": df_ageb, "manzanas": df_manzanas,
    "bancos": df_bancos, "comida": df_comida, "corporativos": df_corporativos,
    "deportes": df_deportes, "negocios": df_negocios,
    "negocios_tiendas": df_negocios_tiendas, "restaurantes": df_restaurantes,
    "tiendas": df_tiendas, "transporte": df_transporte,
}

for nombre, df in tablas.items():
    print(f"{nombre}: {df.count()} filas, {len(df.columns)} columnas")

iter_2020: 666 filas, 286 columnas
iter_2010: 577 filas, 200 columnas
ageburb_2020: 68941 filas, 230 columnas
ageburb_2010: 65721 filas, 198 columnas
municipios: 16 filas, 5 columnas
ageb: 2431 filas, 6 columnas
manzanas: 66789 filas, 9 columnas
bancos: 102568 filas, 41 columnas
comida: 520000 filas, 41 columnas
corporativos: 589 filas, 41 columnas
deportes: 64691 filas, 41 columnas
negocios: 166751 filas, 41 columnas
negocios_tiendas: 73002 filas, 41 columnas
restaurantes: 185532 filas, 41 columnas
tiendas: 608178 filas, 41 columnas
transporte: 39021 filas, 41 columnas


# Casteo

In [0]:
# Schema del ITER 2020

df_iter_2020.printSchema()

root
 |-- ENTIDAD: string (nullable = true)
 |-- NOM_ENT: string (nullable = true)
 |-- MUN: string (nullable = true)
 |-- NOM_MUN: string (nullable = true)
 |-- LOC: string (nullable = true)
 |-- NOM_LOC: string (nullable = true)
 |-- LONGITUD: string (nullable = true)
 |-- LATITUD: string (nullable = true)
 |-- ALTITUD: string (nullable = true)
 |-- POBTOT: string (nullable = true)
 |-- POBFEM: string (nullable = true)
 |-- POBMAS: string (nullable = true)
 |-- P_0A2: string (nullable = true)
 |-- P_0A2_F: string (nullable = true)
 |-- P_0A2_M: string (nullable = true)
 |-- P_3YMAS: string (nullable = true)
 |-- P_3YMAS_F: string (nullable = true)
 |-- P_3YMAS_M: string (nullable = true)
 |-- P_5YMAS: string (nullable = true)
 |-- P_5YMAS_F: string (nullable = true)
 |-- P_5YMAS_M: string (nullable = true)
 |-- P_12YMAS: string (nullable = true)
 |-- P_12YMAS_F: string (nullable = true)
 |-- P_12YMAS_M: string (nullable = true)
 |-- P_15YMAS: string (nullable = true)
 |-- P_15YMAS_F:

In [0]:
from pyspark.sql.functions import col, when

cols_string = {"ENTIDAD", "NOM_ENT", "MUN", "NOM_MUN", "LOC", "NOM_LOC", "TAMLOC"}
cols_double = {"LONGITUD", "LATITUD", "ALTITUD", "REL_H_M", "PROM_HNV",
               "GRAPROES", "GRAPROES_F", "GRAPROES_M", "PROM_OCUP", "PRO_OCUP_C"}

exprs = []
for c in df_iter_2020.columns:
    if c in cols_string:
        exprs.append(col(c))
    elif c in cols_double:
        # "*" -> null explícito antes de castear, en vez de dejar que el cast lo haga solo
        exprs.append(when(col(c) == "*", None).otherwise(col(c)).cast("double").alias(c))
    else:
        exprs.append(when(col(c) == "*", None).otherwise(col(c)).cast("int").alias(c))

df_iter_2020_tipado = df_iter_2020.select(*exprs)

In [0]:
from pyspark.sql.functions import count, when as w

df_iter_2020_tipado.select(
    count(w(col("P_85YMAS_F").isNull(), 1)).alias("nulos_P_85YMAS_F"),
    count(w(col("PCON_DISC").isNull(), 1)).alias("nulos_PCON_DISC"),
    count(w(col("POB_AFRO").isNull(), 1)).alias("nulos_POB_AFRO"),
).show()

+----------------+---------------+--------------+
|nulos_P_85YMAS_F|nulos_PCON_DISC|nulos_POB_AFRO|
+----------------+---------------+--------------+
|             141|            141|           141|
+----------------+---------------+--------------+



In [0]:
fila = df_iter_2020_tipado.select("NOM_ENT").distinct().collect()[0]
texto_corrupto = fila["NOM_ENT"]

print(repr(texto_corrupto))
print([hex(ord(c)) for c in texto_corrupto])

'Ciudad de México'
['0x43', '0x69', '0x75', '0x64', '0x61', '0x64', '0x20', '0x64', '0x65', '0x20', '0x4d', '0xe9', '0x78', '0x69', '0x63', '0x6f']


In [0]:
# Imprimimos el schema del ITER 2010

df_iter_2010.printSchema()

root
 |-- entidad: string (nullable = true)
 |-- nom_ent: string (nullable = true)
 |-- mun: string (nullable = true)
 |-- nom_mun: string (nullable = true)
 |-- loc: string (nullable = true)
 |-- nom_loc: string (nullable = true)
 |-- longitud: string (nullable = true)
 |-- latitud: string (nullable = true)
 |-- altitud: string (nullable = true)
 |-- pobtot: string (nullable = true)
 |-- pobmas: string (nullable = true)
 |-- pobfem: string (nullable = true)
 |-- p_0a2: string (nullable = true)
 |-- p_0a2_m: string (nullable = true)
 |-- p_0a2_f: string (nullable = true)
 |-- p_3ymas: string (nullable = true)
 |-- p_3ymas_m: string (nullable = true)
 |-- p_3ymas_f: string (nullable = true)
 |-- p_5ymas: string (nullable = true)
 |-- p_5ymas_m: string (nullable = true)
 |-- p_5ymas_f: string (nullable = true)
 |-- p_12ymas: string (nullable = true)
 |-- p_12ymas_m: string (nullable = true)
 |-- p_12ymas_f: string (nullable = true)
 |-- p_15ymas: string (nullable = true)
 |-- p_15ymas_m:

In [0]:
from pyspark.sql.functions import col, when

cols_string_2010 = {"entidad", "nom_ent", "mun", "nom_mun", "loc", "nom_loc", "tam_loc"}
cols_double_2010 = {"longitud", "latitud", "altitud", "rel_h_m", "prom_hnv",
                     "graproes", "graproes_m", "graproes_f", "prom_ocup", "pro_ocup_c"}

exprs = []
for c in df_iter_2010.columns:
    if c in cols_string_2010:
        exprs.append(col(c))
    elif c in cols_double_2010:
        exprs.append(when(col(c) == "*", None).otherwise(col(c)).cast("double").alias(c))
    else:
        exprs.append(when(col(c) == "*", None).otherwise(col(c)).cast("int").alias(c))

df_iter_2010_tipado = df_iter_2010.select(*exprs)
df_iter_2010_tipado.printSchema()

root
 |-- entidad: string (nullable = true)
 |-- nom_ent: string (nullable = true)
 |-- mun: string (nullable = true)
 |-- nom_mun: string (nullable = true)
 |-- loc: string (nullable = true)
 |-- nom_loc: string (nullable = true)
 |-- longitud: double (nullable = true)
 |-- latitud: double (nullable = true)
 |-- altitud: double (nullable = true)
 |-- pobtot: integer (nullable = true)
 |-- pobmas: integer (nullable = true)
 |-- pobfem: integer (nullable = true)
 |-- p_0a2: integer (nullable = true)
 |-- p_0a2_m: integer (nullable = true)
 |-- p_0a2_f: integer (nullable = true)
 |-- p_3ymas: integer (nullable = true)
 |-- p_3ymas_m: integer (nullable = true)
 |-- p_3ymas_f: integer (nullable = true)
 |-- p_5ymas: integer (nullable = true)
 |-- p_5ymas_m: integer (nullable = true)
 |-- p_5ymas_f: integer (nullable = true)
 |-- p_12ymas: integer (nullable = true)
 |-- p_12ymas_m: integer (nullable = true)
 |-- p_12ymas_f: integer (nullable = true)
 |-- p_15ymas: integer (nullable = true)


In [0]:
for c in df_iter_2020_tipado.columns:
    df_iter_2020_tipado = df_iter_2020_tipado.withColumnRenamed(c, c.lower())

In [0]:
from pyspark.sql.functions import col, count, when

def resumen_nulos(df, nombre_tabla):
    total = df.count()
    filas = []
    for c in df.columns:
        n_nulos = df.filter(col(c).isNull()).count()
        filas.append((c, n_nulos, round(100 * n_nulos / total, 1)))
    return spark.createDataFrame(filas, ["columna", "nulos", "pct_nulos"]).withColumn("tabla", when(col("columna").isNotNull(), nombre_tabla))

df_nulos_2010 = resumen_nulos(df_iter_2010_tipado, "iter_2010")
display(df_nulos_2010.orderBy(col("nulos").desc()))

columna,nulos,pct_nulos,tabla
rel_h_m,111,19.2,iter_2010
prom_hnv,111,19.2,iter_2010
graproes,111,19.2,iter_2010
graproes_m,111,19.2,iter_2010
graproes_f,111,19.2,iter_2010
prom_ocup,111,19.2,iter_2010
pro_ocup_c,111,19.2,iter_2010
pobmas,98,17.0,iter_2010
pobfem,98,17.0,iter_2010
p_0a2,98,17.0,iter_2010


In [0]:
# Imprimimos el schema del RESAGEBURB

df_ageburb_2020.printSchema()

root
 |-- ENTIDAD: string (nullable = true)
 |-- NOM_ENT: string (nullable = true)
 |-- MUN: string (nullable = true)
 |-- NOM_MUN: string (nullable = true)
 |-- LOC: string (nullable = true)
 |-- NOM_LOC: string (nullable = true)
 |-- AGEB: string (nullable = true)
 |-- MZA: string (nullable = true)
 |-- POBTOT: string (nullable = true)
 |-- POBFEM: string (nullable = true)
 |-- POBMAS: string (nullable = true)
 |-- P_0A2: string (nullable = true)
 |-- P_0A2_F: string (nullable = true)
 |-- P_0A2_M: string (nullable = true)
 |-- P_3YMAS: string (nullable = true)
 |-- P_3YMAS_F: string (nullable = true)
 |-- P_3YMAS_M: string (nullable = true)
 |-- P_5YMAS: string (nullable = true)
 |-- P_5YMAS_F: string (nullable = true)
 |-- P_5YMAS_M: string (nullable = true)
 |-- P_12YMAS: string (nullable = true)
 |-- P_12YMAS_F: string (nullable = true)
 |-- P_12YMAS_M: string (nullable = true)
 |-- P_15YMAS: string (nullable = true)
 |-- P_15YMAS_F: string (nullable = true)
 |-- P_15YMAS_M: stri

In [0]:
df_ageburb_2020.groupBy("MZA").count().orderBy("count", ascending=False).show(10)

+---+-----+
|MZA|count|
+---+-----+
|000| 2485|
|001| 2304|
|002| 2187|
|004| 2148|
|003| 2148|
|005| 2127|
|006| 2111|
|007| 2088|
|008| 2079|
|009| 2060|
+---+-----+
only showing top 10 rows


In [0]:
from pyspark.sql import functions as F

df_ageburb_2020_totales_ageb = df_ageburb_2020.filter(F.col("MZA") == "000")
df_ageburb_2020_manzanas = df_ageburb_2020.filter(F.col("MZA") != "000")

print(f"Totales por AGEB: {df_ageburb_2020_totales_ageb.count()}")
print(f"Manzanas individuales: {df_ageburb_2020_manzanas.count()}")

Totales por AGEB: 2485
Manzanas individuales: 66456


In [0]:
from pyspark.sql.functions import col, when

cols_string = {"ENTIDAD", "NOM_ENT", "MUN", "NOM_MUN", "LOC", "NOM_LOC", "AGEB", "MZA"}
cols_double = {"REL_H_M", "PROM_HNV", "GRAPROES", "GRAPROES_F", "GRAPROES_M",
               "PROM_OCUP", "PRO_OCUP_C"}
valores_nulos_inegi = ["*", "N/D", "N/A"]

exprs = []
for c in df_ageburb_2020_manzanas.columns:
    if c in cols_string:
        exprs.append(col(c))
    elif c in cols_double:
        exprs.append(when(col(c).isin(valores_nulos_inegi), None).otherwise(col(c)).cast("double").alias(c))
    else:
        exprs.append(when(col(c).isin(valores_nulos_inegi), None).otherwise(col(c)).cast("int").alias(c))

df_ageburb_2020_tipado = df_ageburb_2020_manzanas.select(*exprs)
df_ageburb_2020_tipado.printSchema()

root
 |-- ENTIDAD: string (nullable = true)
 |-- NOM_ENT: string (nullable = true)
 |-- MUN: string (nullable = true)
 |-- NOM_MUN: string (nullable = true)
 |-- LOC: string (nullable = true)
 |-- NOM_LOC: string (nullable = true)
 |-- AGEB: string (nullable = true)
 |-- MZA: string (nullable = true)
 |-- POBTOT: integer (nullable = true)
 |-- POBFEM: integer (nullable = true)
 |-- POBMAS: integer (nullable = true)
 |-- P_0A2: integer (nullable = true)
 |-- P_0A2_F: integer (nullable = true)
 |-- P_0A2_M: integer (nullable = true)
 |-- P_3YMAS: integer (nullable = true)
 |-- P_3YMAS_F: integer (nullable = true)
 |-- P_3YMAS_M: integer (nullable = true)
 |-- P_5YMAS: integer (nullable = true)
 |-- P_5YMAS_F: integer (nullable = true)
 |-- P_5YMAS_M: integer (nullable = true)
 |-- P_12YMAS: integer (nullable = true)
 |-- P_12YMAS_F: integer (nullable = true)
 |-- P_12YMAS_M: integer (nullable = true)
 |-- P_15YMAS: integer (nullable = true)
 |-- P_15YMAS_F: integer (nullable = true)
 |--

In [0]:
from pyspark.sql.functions import sum as _sum

def resumen_nulos_rapido(df, nombre_tabla):
    total = df.count()
    agregados = df.select([
        _sum(col(c).isNull().cast("int")).alias(c) for c in df.columns
    ]).collect()[0].asDict()
    filas = [(c, n, round(100 * n / total, 1)) for c, n in agregados.items()]
    return spark.createDataFrame(filas, ["columna", "nulos", "pct_nulos"])

df_nulos_ageburb_2020 = resumen_nulos_rapido(df_ageburb_2020_tipado, "ageburb_2020")
display(df_nulos_ageburb_2020.orderBy(col("nulos").desc()))

columna,nulos,pct_nulos
P15A17A_M,25876,38.9
P15A17A_F,25619,38.6
P_0A2_F,25390,38.2
P_0A2_M,25329,38.1
P_3A5_F,25191,37.9
PCDISC_AUD,25084,37.7
P_3A5_M,24843,37.4
PCLIM_PMEN,24839,37.4
PCDISC_MOT2,24756,37.3
PCDISC_MEN,24474,36.8


In [0]:
# Imprimimos el schema del AGEBURB 2010

df_ageburb_2010.printSchema()

root
 |-- ENTIDAD: string (nullable = true)
 |-- NOM_ENT: string (nullable = true)
 |-- MUN: string (nullable = true)
 |-- NOM_MUN: string (nullable = true)
 |-- LOC: string (nullable = true)
 |-- NOM_LOC: string (nullable = true)
 |-- AGEB: string (nullable = true)
 |-- MZA: string (nullable = true)
 |-- POBTOT: string (nullable = true)
 |-- POBMAS: string (nullable = true)
 |-- POBFEM: string (nullable = true)
 |-- P_0A2: string (nullable = true)
 |-- P_0A2_M: string (nullable = true)
 |-- P_0A2_F: string (nullable = true)
 |-- P_3YMAS: string (nullable = true)
 |-- P_3YMAS_M: string (nullable = true)
 |-- P_3YMAS_F: string (nullable = true)
 |-- P_5YMAS: string (nullable = true)
 |-- P_5YMAS_M: string (nullable = true)
 |-- P_5YMAS_F: string (nullable = true)
 |-- P_12YMAS: string (nullable = true)
 |-- P_12YMAS_M: string (nullable = true)
 |-- P_12YMAS_F: string (nullable = true)
 |-- P_15YMAS: string (nullable = true)
 |-- P_15YMAS_M: string (nullable = true)
 |-- P_15YMAS_F: stri

In [0]:
from pyspark.sql import functions as F

cols_a_revisar = ["POBTOT", "VIVTOT", "TVIVHAB", "POBMAS", "POBFEM", "PROM_HNV"]

for c in cols_a_revisar:
    print(f"--- {c} ---")
    (
        df_ageburb_2010
        .select(c)
        .distinct()
        .filter(~F.col(c).rlike(r'^\d+(\.\d+)?$'))
        .show(20, truncate=False)
    )

--- POBTOT ---
+------+
|POBTOT|
+------+
+------+

--- VIVTOT ---
+------+
|VIVTOT|
+------+
+------+

--- TVIVHAB ---
+-------+
|TVIVHAB|
+-------+
|*      |
+-------+

--- POBMAS ---
+------+
|POBMAS|
+------+
|*     |
+------+

--- POBFEM ---
+------+
|POBFEM|
+------+
|*     |
+------+

--- PROM_HNV ---
+--------+
|PROM_HNV|
+--------+
|*       |
|N/D     |
+--------+



In [0]:
from pyspark.sql import functions as F

id_cols = ["ENTIDAD", "MUN", "LOC", "AGEB", "MZA", "NOM_ENT", "NOM_MUN", "NOM_LOC"]

double_cols = ["REL_H_M", "PROM_HNV", "PROM_OCUP", "PRO_OCUP_C",
               "GRAPROES", "GRAPROES_M", "GRAPROES_F"]

all_cols = df_ageburb_2010.columns
int_cols = [c for c in all_cols if c not in id_cols and c not in double_cols]

valores_raros = set()

for c in int_cols + double_cols:
    rows = (
        df_ageburb_2010
        .select(c)
        .distinct()
        .filter(~F.col(c).rlike(r'^\d+(\.\d+)?$'))
        .filter(F.col(c).isNotNull())
        .collect()
    )
    raros = [row[c] for row in rows]
    if raros:
        print(f"{c}: {raros[:10]}")
        valores_raros.update(raros)

print("\nTodos los valores especiales encontrados en el dataset:")
print(valores_raros)

POBMAS: ['*']
POBFEM: ['*']
P_0A2: ['*', 'N/D']
P_0A2_M: ['*', 'N/D']
P_0A2_F: ['*', 'N/D']
P_3YMAS: ['*', 'N/D']
P_3YMAS_M: ['*', 'N/D']
P_3YMAS_F: ['*', 'N/D']
P_5YMAS: ['*', 'N/D']
P_5YMAS_M: ['*', 'N/D']
P_5YMAS_F: ['*', 'N/D']
P_12YMAS: ['*', 'N/D']
P_12YMAS_M: ['*', 'N/D']
P_12YMAS_F: ['*', 'N/D']
P_15YMAS: ['*', 'N/D']
P_15YMAS_M: ['*', 'N/D']
P_15YMAS_F: ['*', 'N/D']
P_18YMAS: ['*', 'N/D']
P_18YMAS_M: ['*', 'N/D']
P_18YMAS_F: ['*', 'N/D']
P_3A5: ['*', 'N/D']
P_3A5_M: ['*', 'N/D']
P_3A5_F: ['*', 'N/D']
P_6A11: ['*', 'N/D']
P_6A11_M: ['*', 'N/D']
P_6A11_F: ['*', 'N/D']
P_8A14: ['*', 'N/D']
P_8A14_M: ['*', 'N/D']
P_8A14_F: ['*', 'N/D']
P_12A14: ['*', 'N/D']
P_12A14_M: ['*', 'N/D']
P_12A14_F: ['*', 'N/D']
P_15A17: ['*', 'N/D']
P_15A17_M: ['*', 'N/D']
P_15A17_F: ['*', 'N/D']
P_18A24: ['*', 'N/D']
P_18A24_M: ['*', 'N/D']
P_18A24_F: ['*', 'N/D']
P_15A49_F: ['*', 'N/D']
P_60YMAS: ['*', 'N/D']
P_60YMAS_M: ['*', 'N/D']
P_60YMAS_F: ['*', 'N/D']
POB0_14: ['*', 'N/D']
POB15_64: ['*', 'N/D']

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

id_cols = ["ENTIDAD", "MUN", "LOC", "AGEB", "MZA", "NOM_ENT", "NOM_MUN", "NOM_LOC"]

double_cols = ["REL_H_M", "PROM_HNV", "PROM_OCUP", "PRO_OCUP_C",
               "GRAPROES", "GRAPROES_M", "GRAPROES_F"]

all_cols = df_ageburb_2010.columns
int_cols = [c for c in all_cols if c not in id_cols and c not in double_cols]

# Marcadores especiales confirmados para RESAGEBURB 2010 (distinto a 2020, que incluía N/A)
special_values_2010 = ["*", "N/D"]

df_clean_2010 = df_ageburb_2010
for c in int_cols + double_cols:
    df_clean_2010 = df_clean_2010.withColumn(
        c,
        F.when(F.col(c).isin(special_values_2010), None).otherwise(F.col(c))
    )

for c in int_cols:
    df_clean_2010 = df_clean_2010.withColumn(c, F.col(c).cast(IntegerType()))

for c in double_cols:
    df_clean_2010 = df_clean_2010.withColumn(c, F.col(c).cast(DoubleType()))

df_clean_2010.printSchema()

root
 |-- ENTIDAD: string (nullable = true)
 |-- NOM_ENT: string (nullable = true)
 |-- MUN: string (nullable = true)
 |-- NOM_MUN: string (nullable = true)
 |-- LOC: string (nullable = true)
 |-- NOM_LOC: string (nullable = true)
 |-- AGEB: string (nullable = true)
 |-- MZA: string (nullable = true)
 |-- POBTOT: integer (nullable = true)
 |-- POBMAS: integer (nullable = true)
 |-- POBFEM: integer (nullable = true)
 |-- P_0A2: integer (nullable = true)
 |-- P_0A2_M: integer (nullable = true)
 |-- P_0A2_F: integer (nullable = true)
 |-- P_3YMAS: integer (nullable = true)
 |-- P_3YMAS_M: integer (nullable = true)
 |-- P_3YMAS_F: integer (nullable = true)
 |-- P_5YMAS: integer (nullable = true)
 |-- P_5YMAS_M: integer (nullable = true)
 |-- P_5YMAS_F: integer (nullable = true)
 |-- P_12YMAS: integer (nullable = true)
 |-- P_12YMAS_M: integer (nullable = true)
 |-- P_12YMAS_F: integer (nullable = true)
 |-- P_15YMAS: integer (nullable = true)
 |-- P_15YMAS_M: integer (nullable = true)
 |--

In [0]:
from pyspark.sql import functions as F

total_filas = df_clean_2010.count()

nulos_pct = df_clean_2010.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total_filas * 100).alias(c)
    for c in int_cols + double_cols
])

# Transponer para verlo como tabla columna -> % nulo
nulos_dict = nulos_pct.collect()[0].asDict()
nulos_ordenado = sorted(nulos_dict.items(), key=lambda x: x[1], reverse=True)

for col, pct in nulos_ordenado:
    print(f"{col}: {pct:.1f}%")

VIVPAR_UT: 92.1%
VIVPAR_DES: 92.0%
TVIVPARHAB: 92.0%
TVIVPAR: 92.0%
VIVPAR_HAB: 92.0%
TVIVHAB: 92.0%
P15A17A_M: 37.3%
P15A17A_F: 37.2%
P15SEC_INF: 36.1%
PDESOCUP_M: 35.1%
P_0A2_F: 35.0%
P_12A14_F: 34.6%
P_12A14_M: 34.5%
P_0A2_M: 34.5%
P_3A5_F: 34.4%
P_15A17_F: 34.0%
P_3A5_M: 34.0%
P15SEC_INM: 33.7%
PDESOCUP_F: 33.6%
P_15A17_M: 33.5%
P15YM_SE_F: 33.2%
P18A24A_F: 33.1%
P3A5_NOA_M: 32.7%
P3A5_NOA: 32.7%
P18A24A_M: 32.5%
P15YM_AN_F: 32.5%
P3A5_NOA_F: 32.2%
PDESOCUP: 31.9%
PCLIM_MOT: 31.6%
P15PRI_INM: 30.8%
PCLIM_VIS: 30.7%
PCLIM_MEN2: 30.7%
P15YM_AN: 30.5%
P15YM_SE_M: 30.1%
P15YM_SE: 29.9%
P15PRI_INF: 29.8%
PCLIM_AUD: 29.2%
P15SEC_IN: 28.4%
P5_HLI: 28.3%
P3YM_HLI: 28.3%
PRESOE05_M: 28.2%
PRESOE05_F: 28.0%
P3YM_HLI_F: 27.7%
P15YM_AN_M: 26.3%
P3YM_HLI_M: 26.1%
P15A17A: 26.0%
P_0A2: 25.7%
P5_HLI_HE: 25.6%
P3HLI_HE: 25.6%
P3HLI_HE_F: 25.1%
P15PRI_COM: 25.0%
PCON_LIM: 24.9%
PCLIM_LENG: 24.9%
P_3A5: 24.5%
P_6A11_F: 24.0%
P15PRI_IN: 23.7%
P_12A14: 23.7%
P_6A11_M: 23.5%
PRESOE05: 23.2%
P3HLI_HE_M:

In [0]:
# Ver cuántas filas tienen valor real vs "*"/"N/D" vs null puro en estas columnas
cols_sospechosas = ["TVIVHAB", "VIVPAR_HAB", "TVIVPAR", "TVIVPARHAB", "VIVPAR_DES", "VIVPAR_UT"]

for c in cols_sospechosas:
    print(f"--- {c} ---")
    df_ageburb_2010.groupBy(c).count().orderBy(F.desc("count")).show(10, truncate=False)

--- TVIVHAB ---
+-------+-----+
|TVIVHAB|count|
+-------+-----+
|*      |60469|
|0      |2798 |
|672    |6    |
|875    |6    |
|604    |5    |
|829    |5    |
|996    |5    |
|504    |5    |
|848    |5    |
|565    |5    |
+-------+-----+
only showing top 10 rows
--- VIVPAR_HAB ---
+----------+-----+
|VIVPAR_HAB|count|
+----------+-----+
|*         |60472|
|0         |2799 |
|758       |7    |
|920       |7    |
|595       |6    |
|1300      |6    |
|451       |6    |
|842       |6    |
|859       |6    |
|634       |5    |
+----------+-----+
only showing top 10 rows
--- TVIVPAR ---
+-------+-----+
|TVIVPAR|count|
+-------+-----+
|*      |60472|
|0      |2798 |
|956    |7    |
|961    |7    |
|630    |6    |
|596    |6    |
|1122   |6    |
|1083   |6    |
|553    |6    |
|1324   |5    |
+-------+-----+
only showing top 10 rows
--- TVIVPARHAB ---
+----------+-----+
|TVIVPARHAB|count|
+----------+-----+
|*         |60473|
|0         |2798 |
|733       |7    |
|565       |6    |
|829    

In [0]:
# Imprimimos el schema del Marco geoestadístico: municipios

df_municipios.printSchema()

root
 |-- CVEGEO: string (nullable = true)
 |-- CVE_ENT: string (nullable = true)
 |-- CVE_MUN: string (nullable = true)
 |-- NOMGEO: string (nullable = true)
 |-- geometry: string (nullable = true)



In [0]:
from pyspark.sql import functions as F

# 1. Nulos por columna
total_filas = df_municipios.count()
print(f"Total filas: {total_filas}")

df_municipios.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total_filas * 100).alias(c)
    for c in df_municipios.columns
]).show()

# 2. Duplicados en la clave CVEGEO (no deberían existir)
duplicados = (
    df_municipios.groupBy("CVEGEO")
    .count()
    .filter(F.col("count") > 1)
)
print(f"CVEGEO duplicados: {duplicados.count()}")
duplicados.show()

# 3. Confirmar que CVEGEO = CVE_ENT + CVE_MUN concatenado (con padding correcto)
df_check = df_municipios.withColumn(
    "CVEGEO_calculado",
    F.concat(F.lpad(F.col("CVE_ENT"), 2, "0"), F.lpad(F.col("CVE_MUN"), 3, "0"))
)

inconsistencias = df_check.filter(F.col("CVEGEO") != F.col("CVEGEO_calculado"))
print(f"Filas donde CVEGEO no coincide con CVE_ENT+CVE_MUN: {inconsistencias.count()}")
inconsistencias.select("CVEGEO", "CVE_ENT", "CVE_MUN", "CVEGEO_calculado").show(20, truncate=False)

# 4. Longitud esperada de cada campo (CVE_ENT=2 dígitos, CVE_MUN=3 dígitos, CVEGEO=5 dígitos)
df_municipios.select(
    F.length("CVE_ENT").alias("len_ent"),
    F.length("CVE_MUN").alias("len_mun"),
    F.length("CVEGEO").alias("len_cvegeo")
).distinct().show()

# 5. Confirmar cobertura: deberían ser las 16 alcaldías de CDMX
df_municipios.select("NOMGEO").distinct().orderBy("NOMGEO").show(16, truncate=False)

Total filas: 16
+------+-------+-------+------+--------+
|CVEGEO|CVE_ENT|CVE_MUN|NOMGEO|geometry|
+------+-------+-------+------+--------+
|   0.0|    0.0|    0.0|   0.0|     0.0|
+------+-------+-------+------+--------+

CVEGEO duplicados: 0
+------+-----+
|CVEGEO|count|
+------+-----+
+------+-----+

Filas donde CVEGEO no coincide con CVE_ENT+CVE_MUN: 0
+------+-------+-------+----------------+
|CVEGEO|CVE_ENT|CVE_MUN|CVEGEO_calculado|
+------+-------+-------+----------------+
+------+-------+-------+----------------+

+-------+-------+----------+
|len_ent|len_mun|len_cvegeo|
+-------+-------+----------+
|      2|      3|         5|
+-------+-------+----------+

+----------------------+
|NOMGEO                |
+----------------------+
|Azcapotzalco          |
|Benito Juárez         |
|Coyoacán              |
|Cuajimalpa de Morelos |
|Cuauhtémoc            |
|Gustavo A. Madero     |
|Iztacalco             |
|Iztapalapa            |
|La Magdalena Contreras|
|Miguel Hidalgo        |
|M

In [0]:
# Imprimos el schema del Marco geoestadístico: manzanas

df_ageb.printSchema()

root
 |-- CVEGEO: string (nullable = true)
 |-- CVE_ENT: string (nullable = true)
 |-- CVE_MUN: string (nullable = true)
 |-- CVE_LOC: string (nullable = true)
 |-- CVE_AGEB: string (nullable = true)
 |-- geometry: string (nullable = true)



In [0]:
from pyspark.sql import functions as F

total_filas = df_ageb.count()
print(f"Total filas: {total_filas}")

# 1. Nulos por columna
df_ageb.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total_filas * 100).alias(c)
    for c in df_ageb.columns
]).show()

# 2. Duplicados en CVEGEO (clave a nivel AGEB)
duplicados = (
    df_ageb.groupBy("CVEGEO")
    .count()
    .filter(F.col("count") > 1)
)
print(f"CVEGEO duplicados: {duplicados.count()}")
duplicados.show()

# 3. Confirmar que CVEGEO = CVE_ENT + CVE_MUN + CVE_LOC + CVE_AGEB concatenado
df_check = df_ageb.withColumn(
    "CVEGEO_calculado",
    F.concat(
        F.lpad(F.col("CVE_ENT"), 2, "0"),
        F.lpad(F.col("CVE_MUN"), 3, "0"),
        F.lpad(F.col("CVE_LOC"), 4, "0"),
        F.lpad(F.col("CVE_AGEB"), 4, "0")
    )
)

inconsistencias = df_check.filter(F.col("CVEGEO") != F.col("CVEGEO_calculado"))
print(f"Filas donde CVEGEO no coincide: {inconsistencias.count()}")
inconsistencias.select("CVEGEO", "CVE_ENT", "CVE_MUN", "CVE_LOC", "CVE_AGEB", "CVEGEO_calculado").show(20, truncate=False)

# 4. Longitudes esperadas por campo
df_ageb.select(
    F.length("CVE_ENT").alias("len_ent"),
    F.length("CVE_MUN").alias("len_mun"),
    F.length("CVE_LOC").alias("len_loc"),
    F.length("CVE_AGEB").alias("len_ageb"),
    F.length("CVEGEO").alias("len_cvegeo")
).distinct().show()

# 5. Total de AGEB únicos por municipio (para comparar contra tu conteo de 2,431 AGEB ya conocido)
df_ageb.groupBy("CVE_MUN").agg(F.countDistinct("CVE_AGEB").alias("ageb_unicos")).orderBy("CVE_MUN").show(16)

Total filas: 2431
+------+-------+-------+-------+--------+--------+
|CVEGEO|CVE_ENT|CVE_MUN|CVE_LOC|CVE_AGEB|geometry|
+------+-------+-------+-------+--------+--------+
|   0.0|    0.0|    0.0|    0.0|     0.0|     0.0|
+------+-------+-------+-------+--------+--------+

CVEGEO duplicados: 0
+------+-----+
|CVEGEO|count|
+------+-----+
+------+-----+

Filas donde CVEGEO no coincide: 0
+------+-------+-------+-------+--------+----------------+
|CVEGEO|CVE_ENT|CVE_MUN|CVE_LOC|CVE_AGEB|CVEGEO_calculado|
+------+-------+-------+-------+--------+----------------+
+------+-------+-------+-------+--------+----------------+

+-------+-------+-------+--------+----------+
|len_ent|len_mun|len_loc|len_ageb|len_cvegeo|
+-------+-------+-------+--------+----------+
|      2|      3|      4|       4|        13|
+-------+-------+-------+--------+----------+

+-------+-----------+
|CVE_MUN|ageb_unicos|
+-------+-----------+
|    002|        103|
|    003|        156|
|    004|         31|
|    005| 

In [0]:
# Imprimimos el schema del Marco geoestadístico: manzanas

df_manzanas.printSchema()

root
 |-- CVEGEO: string (nullable = true)
 |-- CVE_ENT: string (nullable = true)
 |-- CVE_MUN: string (nullable = true)
 |-- CVE_LOC: string (nullable = true)
 |-- CVE_AGEB: string (nullable = true)
 |-- CVE_MZA: string (nullable = true)
 |-- AMBITO: string (nullable = true)
 |-- TIPOMZA: string (nullable = true)
 |-- geometry: string (nullable = true)



In [0]:
from pyspark.sql import functions as F

total_filas = df_manzanas.count()
print(f"Total filas: {total_filas}")

# 1. Nulos por columna
df_manzanas.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total_filas * 100).alias(c)
    for c in df_manzanas.columns
]).show()

# 2. Duplicados en CVEGEO (clave a nivel manzana)
duplicados = (
    df_manzanas.groupBy("CVEGEO")
    .count()
    .filter(F.col("count") > 1)
)
print(f"CVEGEO duplicados: {duplicados.count()}")
duplicados.show()

# 3. Longitudes de cada campo (para el lpad correcto de la clave)
df_manzanas.select(
    F.length("CVE_ENT").alias("len_ent"),
    F.length("CVE_MUN").alias("len_mun"),
    F.length("CVE_LOC").alias("len_loc"),
    F.length("CVE_AGEB").alias("len_ageb"),
    F.length("CVE_MZA").alias("len_mza"),
    F.length("CVEGEO").alias("len_cvegeo")
).distinct().show()

# 4. Confirmar que CVEGEO = concatenación (usando los anchos que salgan de (3))
# Ejemplo si CVE_MZA resulta ser 3 dígitos (ajusta según el resultado real):
df_check = df_manzanas.withColumn(
    "CVEGEO_calculado",
    F.concat(
        F.lpad(F.col("CVE_ENT"), 2, "0"),
        F.lpad(F.col("CVE_MUN"), 3, "0"),
        F.lpad(F.col("CVE_LOC"), 4, "0"),
        F.lpad(F.col("CVE_AGEB"), 4, "0"),
        F.lpad(F.col("CVE_MZA"), 3, "0")
    )
)
inconsistencias = df_check.filter(F.col("CVEGEO") != F.col("CVEGEO_calculado"))
print(f"Filas donde CVEGEO no coincide: {inconsistencias.count()}")
inconsistencias.select("CVEGEO", "CVE_ENT", "CVE_MUN", "CVE_LOC", "CVE_AGEB", "CVE_MZA", "CVEGEO_calculado").show(20, truncate=False)

# 5. Total de manzanas por municipio (comparar contra tu conteo ya confirmado: 66,789)
df_manzanas.groupBy("CVE_MUN").count().orderBy("CVE_MUN").show(16)

# 6. Valores únicos de las columnas categóricas nuevas
df_manzanas.groupBy("AMBITO").count().show()
df_manzanas.groupBy("TIPOMZA").count().show()

Total filas: 66789
+------+-------+-------+-------+--------+-------+------+-------+--------+
|CVEGEO|CVE_ENT|CVE_MUN|CVE_LOC|CVE_AGEB|CVE_MZA|AMBITO|TIPOMZA|geometry|
+------+-------+-------+-------+--------+-------+------+-------+--------+
|   0.0|    0.0|    0.0|    0.0|     0.0|    0.0|   0.0|    0.0|     0.0|
+------+-------+-------+-------+--------+-------+------+-------+--------+

CVEGEO duplicados: 0
+------+-----+
|CVEGEO|count|
+------+-----+
+------+-----+

+-------+-------+-------+--------+-------+----------+
|len_ent|len_mun|len_loc|len_ageb|len_mza|len_cvegeo|
+-------+-------+-------+--------+-------+----------+
|      2|      3|      4|       4|      3|        16|
+-------+-------+-------+--------+-------+----------+

Filas donde CVEGEO no coincide: 0
+------+-------+-------+-------+--------+-------+----------------+
|CVEGEO|CVE_ENT|CVE_MUN|CVE_LOC|CVE_AGEB|CVE_MZA|CVEGEO_calculado|
+------+-------+-------+-------+--------+-------+----------------+
+------+-------+------

In [0]:
# ¿Cuántas manzanas "no habitables por diseño" hay?
no_habitacionales = df_manzanas.filter(
    F.col("TIPOMZA").isin(["Glorieta", "Bajo Puente", "Parque o Jardín", "Camellón"])
)
print(f"Manzanas no habitacionales por tipo: {no_habitacionales.count()}")

Manzanas no habitacionales por tipo: 541


In [0]:
from pyspark.sql import functions as F

manzanas_excluir = df_manzanas.filter(
    (F.col("AMBITO") == "Rural") |
    (F.col("TIPOMZA").isin(["Glorieta", "Bajo Puente", "Parque o Jardín", "Camellón"]))
)
print(f"Total manzanas a excluir del análisis de densidad: {manzanas_excluir.count()}")

df_manzanas_analisis = df_manzanas.subtract(manzanas_excluir)
print(f"Manzanas que quedan para el análisis: {df_manzanas_analisis.count()}")

Total manzanas a excluir del análisis de densidad: 949
Manzanas que quedan para el análisis: 65840


In [0]:
# Imprimimos el schema del DENUE: Bancos

df_bancos.printSchema()

root
 |-- id: string (nullable = true)
 |-- nom_estab: string (nullable = true)
 |-- raz_social: string (nullable = true)
 |-- codigo_act: string (nullable = true)
 |-- nombre_act: string (nullable = true)
 |-- per_ocu: string (nullable = true)
 |-- tipo_vial: string (nullable = true)
 |-- nom_vial: string (nullable = true)
 |-- tipo_v_e_1: string (nullable = true)
 |-- nom_v_e_1: string (nullable = true)
 |-- tipo_v_e_2: string (nullable = true)
 |-- nom_v_e_2: string (nullable = true)
 |-- tipo_v_e_3: string (nullable = true)
 |-- nom_v_e_3: string (nullable = true)
 |-- numero_ext: string (nullable = true)
 |-- letra_ext: string (nullable = true)
 |-- edificio: string (nullable = true)
 |-- edificio_e: string (nullable = true)
 |-- numero_int: string (nullable = true)
 |-- letra_int: string (nullable = true)
 |-- tipo_asent: string (nullable = true)
 |-- nomb_asent: string (nullable = true)
 |-- tipoCenCom: string (nullable = true)
 |-- nom_CenCom: string (nullable = true)
 |-- num_

In [0]:
df_bancos.select("fecha_alta").distinct().show(10, truncate=False)

+----------+
|fecha_alta|
+----------+
|2020-04   |
|2019-11   |
|2010-07   |
|2014-12   |
|2020-11   |
|2011-03   |
|2017-11   |
|2019-04   |
|2013-07   |
|2016-01   |
+----------+
only showing top 10 rows


In [0]:
df_bancos.filter(~F.col("fecha_alta").rlike(r"^\d{4}[-\s]\d{2}\s*$")).select("fecha_alta").distinct().show(50, truncate=False)

+----------+
|fecha_alta|
+----------+
|19.434336 |
|19.430709 |
|Fijo      |
|-98.2026  |
+----------+



In [0]:
filas_sospechosas = df_bancos.filter(
    ~F.col("fecha_alta").rlike(r"^\d{4}[-\s]\d{2}\s*$")
)
print(f"Filas con fecha_alta corrupta: {filas_sospechosas.count()}")

# Ver la fila COMPLETA de una de ellas, no solo la columna fecha_alta
filas_sospechosas.show(5, truncate=False, vertical=True)

Filas con fecha_alta corrupta: 4
-RECORD 0-----------------------------------------------------
 id         | 6901284                                         
 nom_estab  | BANORTE/IXE                                     
 raz_social | BANCO BANORTE IXE                               
 codigo_act | 522110                                          
 nombre_act | Banca m�ltiple                                  
 per_ocu    | 0 a 5 personas                                  
 tipo_vial  | CALLE                                           
 nom_vial   | "=""AEROPUERTO�INTERNACIONAL�DE�LA�CD�DE�MEXICO 
 tipo_v_e_1 | �LOCALES�25�Y�26                                
 nom_v_e_1  | �SALA�"""                                       
 tipo_v_e_2 | NULL                                            
 nom_v_e_2  | NULL                                            
 tipo_v_e_3 | NULL                                            
 nom_v_e_3  | NULL                                            
 numero_ext | NULL    

In [0]:
ids_corruptos = [6901284, 6901277, 6901333, 6901557]

print(f"Filas antes: {df_bancos.count()}")
df_bancos_clean = df_bancos.filter(~F.col("id").isin(ids_corruptos))
print(f"Filas después: {df_bancos_clean.count()}")

# Ahora sí aplicar el casteo normal, ya sin las filas problemáticas
df_bancos_clean = df_bancos_clean \
    .withColumn("latitud", F.col("latitud").cast(DoubleType())) \
    .withColumn("longitud", F.col("longitud").cast(DoubleType())) \
    .withColumn("fecha_alta", F.to_date(F.trim(F.col("fecha_alta")), "yyyy-MM"))

Filas antes: 102568
Filas después: 102564


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

ids_corruptos = [6901284, 6901277, 6901333, 6901557]

df_bancos_clean = (
    df_bancos
    .filter(~F.col("id").isin(ids_corruptos))
    .withColumn("latitud", F.col("latitud").cast(DoubleType()))
    .withColumn("longitud", F.col("longitud").cast(DoubleType()))
    .withColumn(
        "fecha_alta_norm",
        F.regexp_replace(F.trim(F.col("fecha_alta")), r"[^0-9]", "-")
    )
    .withColumn("fecha_alta", F.expr("try_to_date(fecha_alta_norm, 'yyyy-MM')"))
    .drop("fecha_alta_norm")
)

total = df_bancos_clean.count()
print(f"Filas: {total}")

df_bancos_clean.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total * 100).alias(c)
    for c in df_bancos_clean.columns
]).show(vertical=True, truncate=False)

Filas: 102564
-RECORD 0--------------------------
 id         | 0.0                  
 nom_estab  | 9.750009750009749E-4 
 raz_social | 6.8181818181818175   
 codigo_act | 0.0                  
 nombre_act | 0.0                  
 per_ocu    | 0.0                  
 tipo_vial  | 3.237003237003237    
 nom_vial   | 0.0351000351000351   
 tipo_v_e_1 | 52.95425295425296    
 nom_v_e_1  | 50.690300690300695   
 tipo_v_e_2 | 52.82360282360282    
 nom_v_e_2  | 50.712725712725714   
 tipo_v_e_3 | 52.84310284310284    
 nom_v_e_3  | 50.76830076830077    
 numero_ext | 55.82173082173082    
 letra_ext  | 83.14320814320814    
 edificio   | 92.6026676026676     
 edificio_e | 92.30041730041731    
 numero_int | 44.325494325494326   
 letra_int  | 80.45805545805545    
 tipo_asent | 2.298077298077298    
 nomb_asent | 0.0702000702000702   
 tipoCenCom | 90.78819078819079    
 nom_CenCom | 90.68679068679069    
 num_local  | 89.42611442611442    
 cod_postal | 3.2116532116532115   
 cve_ent    | 

In [0]:
# Imprimimos el dataframe de comida

df_comida.printSchema()

root
 |-- id: string (nullable = true)
 |-- nom_estab: string (nullable = true)
 |-- raz_social: string (nullable = true)
 |-- codigo_act: string (nullable = true)
 |-- nombre_act: string (nullable = true)
 |-- per_ocu: string (nullable = true)
 |-- tipo_vial: string (nullable = true)
 |-- nom_vial: string (nullable = true)
 |-- tipo_v_e_1: string (nullable = true)
 |-- nom_v_e_1: string (nullable = true)
 |-- tipo_v_e_2: string (nullable = true)
 |-- nom_v_e_2: string (nullable = true)
 |-- tipo_v_e_3: string (nullable = true)
 |-- nom_v_e_3: string (nullable = true)
 |-- numero_ext: string (nullable = true)
 |-- letra_ext: string (nullable = true)
 |-- edificio: string (nullable = true)
 |-- edificio_e: string (nullable = true)
 |-- numero_int: string (nullable = true)
 |-- letra_int: string (nullable = true)
 |-- tipo_asent: string (nullable = true)
 |-- nomb_asent: string (nullable = true)
 |-- tipoCenCom: string (nullable = true)
 |-- nom_CenCom: string (nullable = true)
 |-- num_

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

def limpiar_denue(df, nombre="df"):
    total_inicial = df.count()
    
    # 1. Revisar filas con fecha_alta corrupta (posible desfase de columnas)
    corruptas = df.filter(~F.col("fecha_alta").rlike(r"^\d{4}[-\s/]\d{2}\s*$"))
    n_corruptas = corruptas.count()
    if n_corruptas > 0:
        print(f"[{nombre}] {n_corruptas} filas sospechosas de corrupción de columnas:")
        corruptas.select("id", "nom_estab", "fecha_alta").show(n_corruptas, truncate=False)
    
    ids_corruptos = [row["id"] for row in corruptas.select("id").collect()]
    
    # 2. Excluir corruptas y castear
    df_clean = (
        df
        .filter(~F.col("id").isin(ids_corruptos))
        .withColumn("latitud", F.col("latitud").cast(DoubleType()))
        .withColumn("longitud", F.col("longitud").cast(DoubleType()))
        .withColumn(
            "fecha_alta_norm",
            F.regexp_replace(F.trim(F.col("fecha_alta")), r"[^0-9]", "-")
        )
        .withColumn("fecha_alta", F.expr("try_to_date(fecha_alta_norm, 'yyyy-MM')"))
        .drop("fecha_alta_norm")
    )
    
    print(f"[{nombre}] Filas: {total_inicial} -> {df_clean.count()}")
    return df_clean

df_comida_clean = limpiar_denue(df_comida, "df_comida")

[df_comida] Filas: 520000 -> 520000


In [0]:
total = df_comida_clean.count()
cols_criticas = ["codigo_act", "nombre_act", "cve_mun", "cve_loc", "ageb", "manzana", "latitud", "longitud", "fecha_alta"]

df_comida_clean.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total * 100).alias(c)
    for c in cols_criticas
]).show(vertical=True, truncate=False)

-RECORD 0---------
 codigo_act | 0.0 
 nombre_act | 0.0 
 cve_mun    | 0.0 
 cve_loc    | 0.0 
 ageb       | 0.0 
 manzana    | 0.0 
 latitud    | 0.0 
 longitud   | 0.0 
 fecha_alta | 0.0 



In [0]:
# Imprimimos el schema del DataFrame Denue Corporativos

df_corporativos.printSchema()

root
 |-- id: string (nullable = true)
 |-- nom_estab: string (nullable = true)
 |-- raz_social: string (nullable = true)
 |-- codigo_act: string (nullable = true)
 |-- nombre_act: string (nullable = true)
 |-- per_ocu: string (nullable = true)
 |-- tipo_vial: string (nullable = true)
 |-- nom_vial: string (nullable = true)
 |-- tipo_v_e_1: string (nullable = true)
 |-- nom_v_e_1: string (nullable = true)
 |-- tipo_v_e_2: string (nullable = true)
 |-- nom_v_e_2: string (nullable = true)
 |-- tipo_v_e_3: string (nullable = true)
 |-- nom_v_e_3: string (nullable = true)
 |-- numero_ext: string (nullable = true)
 |-- letra_ext: string (nullable = true)
 |-- edificio: string (nullable = true)
 |-- edificio_e: string (nullable = true)
 |-- numero_int: string (nullable = true)
 |-- letra_int: string (nullable = true)
 |-- tipo_asent: string (nullable = true)
 |-- nomb_asent: string (nullable = true)
 |-- tipoCenCom: string (nullable = true)
 |-- nom_CenCom: string (nullable = true)
 |-- num_

In [0]:
df_corporativos_clean = limpiar_denue(df_corporativos, "df_corporativos")

df_corporativos_clean.printSchema()

[df_corporativos] Filas: 589 -> 589
root
 |-- id: string (nullable = true)
 |-- nom_estab: string (nullable = true)
 |-- raz_social: string (nullable = true)
 |-- codigo_act: string (nullable = true)
 |-- nombre_act: string (nullable = true)
 |-- per_ocu: string (nullable = true)
 |-- tipo_vial: string (nullable = true)
 |-- nom_vial: string (nullable = true)
 |-- tipo_v_e_1: string (nullable = true)
 |-- nom_v_e_1: string (nullable = true)
 |-- tipo_v_e_2: string (nullable = true)
 |-- nom_v_e_2: string (nullable = true)
 |-- tipo_v_e_3: string (nullable = true)
 |-- nom_v_e_3: string (nullable = true)
 |-- numero_ext: string (nullable = true)
 |-- letra_ext: string (nullable = true)
 |-- edificio: string (nullable = true)
 |-- edificio_e: string (nullable = true)
 |-- numero_int: string (nullable = true)
 |-- letra_int: string (nullable = true)
 |-- tipo_asent: string (nullable = true)
 |-- nomb_asent: string (nullable = true)
 |-- tipoCenCom: string (nullable = true)
 |-- nom_CenCom

In [0]:
total = df_corporativos_clean.count()
cols_criticas = ["codigo_act", "nombre_act", "cve_mun", "cve_loc", "ageb", "manzana", "latitud", "longitud", "fecha_alta"]

df_corporativos_clean.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total * 100).alias(c)
    for c in cols_criticas
]).show(vertical=True, truncate=False)

-RECORD 0---------
 codigo_act | 0.0 
 nombre_act | 0.0 
 cve_mun    | 0.0 
 cve_loc    | 0.0 
 ageb       | 0.0 
 manzana    | 0.0 
 latitud    | 0.0 
 longitud   | 0.0 
 fecha_alta | 0.0 



In [0]:
# Imprimos el schema de deportes

df_deportes.printSchema()

root
 |-- id: string (nullable = true)
 |-- nom_estab: string (nullable = true)
 |-- raz_social: string (nullable = true)
 |-- codigo_act: string (nullable = true)
 |-- nombre_act: string (nullable = true)
 |-- per_ocu: string (nullable = true)
 |-- tipo_vial: string (nullable = true)
 |-- nom_vial: string (nullable = true)
 |-- tipo_v_e_1: string (nullable = true)
 |-- nom_v_e_1: string (nullable = true)
 |-- tipo_v_e_2: string (nullable = true)
 |-- nom_v_e_2: string (nullable = true)
 |-- tipo_v_e_3: string (nullable = true)
 |-- nom_v_e_3: string (nullable = true)
 |-- numero_ext: string (nullable = true)
 |-- letra_ext: string (nullable = true)
 |-- edificio: string (nullable = true)
 |-- edificio_e: string (nullable = true)
 |-- numero_int: string (nullable = true)
 |-- letra_int: string (nullable = true)
 |-- tipo_asent: string (nullable = true)
 |-- nomb_asent: string (nullable = true)
 |-- tipoCenCom: string (nullable = true)
 |-- nom_CenCom: string (nullable = true)
 |-- num_

In [0]:
df_deportes_clean = limpiar_denue(df_deportes, "df_deportes")

df_deportes_clean.printSchema()

[df_deportes] Filas: 64691 -> 64691
root
 |-- id: string (nullable = true)
 |-- nom_estab: string (nullable = true)
 |-- raz_social: string (nullable = true)
 |-- codigo_act: string (nullable = true)
 |-- nombre_act: string (nullable = true)
 |-- per_ocu: string (nullable = true)
 |-- tipo_vial: string (nullable = true)
 |-- nom_vial: string (nullable = true)
 |-- tipo_v_e_1: string (nullable = true)
 |-- nom_v_e_1: string (nullable = true)
 |-- tipo_v_e_2: string (nullable = true)
 |-- nom_v_e_2: string (nullable = true)
 |-- tipo_v_e_3: string (nullable = true)
 |-- nom_v_e_3: string (nullable = true)
 |-- numero_ext: string (nullable = true)
 |-- letra_ext: string (nullable = true)
 |-- edificio: string (nullable = true)
 |-- edificio_e: string (nullable = true)
 |-- numero_int: string (nullable = true)
 |-- letra_int: string (nullable = true)
 |-- tipo_asent: string (nullable = true)
 |-- nomb_asent: string (nullable = true)
 |-- tipoCenCom: string (nullable = true)
 |-- nom_CenCom

In [0]:
total = df_deportes_clean.count()
cols_criticas = ["codigo_act", "nombre_act", "cve_mun", "cve_loc", "ageb", "manzana", "latitud", "longitud", "fecha_alta"]

df_deportes_clean.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total * 100).alias(c)
    for c in cols_criticas
]).show(vertical=True, truncate=False)

-RECORD 0---------
 codigo_act | 0.0 
 nombre_act | 0.0 
 cve_mun    | 0.0 
 cve_loc    | 0.0 
 ageb       | 0.0 
 manzana    | 0.0 
 latitud    | 0.0 
 longitud   | 0.0 
 fecha_alta | 0.0 



In [0]:
df_negocios.printSchema()

root
 |-- id: string (nullable = true)
 |-- nom_estab: string (nullable = true)
 |-- raz_social: string (nullable = true)
 |-- codigo_act: string (nullable = true)
 |-- nombre_act: string (nullable = true)
 |-- per_ocu: string (nullable = true)
 |-- tipo_vial: string (nullable = true)
 |-- nom_vial: string (nullable = true)
 |-- tipo_v_e_1: string (nullable = true)
 |-- nom_v_e_1: string (nullable = true)
 |-- tipo_v_e_2: string (nullable = true)
 |-- nom_v_e_2: string (nullable = true)
 |-- tipo_v_e_3: string (nullable = true)
 |-- nom_v_e_3: string (nullable = true)
 |-- numero_ext: string (nullable = true)
 |-- letra_ext: string (nullable = true)
 |-- edificio: string (nullable = true)
 |-- edificio_e: string (nullable = true)
 |-- numero_int: string (nullable = true)
 |-- letra_int: string (nullable = true)
 |-- tipo_asent: string (nullable = true)
 |-- nomb_asent: string (nullable = true)
 |-- tipoCenCom: string (nullable = true)
 |-- nom_CenCom: string (nullable = true)
 |-- num_

In [0]:
df_negocios_clean = limpiar_denue(df_negocios, "df_negocios")

df_negocios_clean.printSchema()

[df_negocios] Filas: 166751 -> 166751
root
 |-- id: string (nullable = true)
 |-- nom_estab: string (nullable = true)
 |-- raz_social: string (nullable = true)
 |-- codigo_act: string (nullable = true)
 |-- nombre_act: string (nullable = true)
 |-- per_ocu: string (nullable = true)
 |-- tipo_vial: string (nullable = true)
 |-- nom_vial: string (nullable = true)
 |-- tipo_v_e_1: string (nullable = true)
 |-- nom_v_e_1: string (nullable = true)
 |-- tipo_v_e_2: string (nullable = true)
 |-- nom_v_e_2: string (nullable = true)
 |-- tipo_v_e_3: string (nullable = true)
 |-- nom_v_e_3: string (nullable = true)
 |-- numero_ext: string (nullable = true)
 |-- letra_ext: string (nullable = true)
 |-- edificio: string (nullable = true)
 |-- edificio_e: string (nullable = true)
 |-- numero_int: string (nullable = true)
 |-- letra_int: string (nullable = true)
 |-- tipo_asent: string (nullable = true)
 |-- nomb_asent: string (nullable = true)
 |-- tipoCenCom: string (nullable = true)
 |-- nom_CenC

In [0]:
total = df_negocios_clean.count()
cols_criticas = ["codigo_act", "nombre_act", "cve_mun", "cve_loc", "ageb", "manzana", "latitud", "longitud", "fecha_alta"]

df_negocios_clean.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total * 100).alias(c)
    for c in cols_criticas
]).show(vertical=True, truncate=False)

-RECORD 0---------
 codigo_act | 0.0 
 nombre_act | 0.0 
 cve_mun    | 0.0 
 cve_loc    | 0.0 
 ageb       | 0.0 
 manzana    | 0.0 
 latitud    | 0.0 
 longitud   | 0.0 
 fecha_alta | 0.0 



In [0]:
df_negocios_clean = limpiar_denue(df_negocios, "df_negocios")

df_negocios_clean.printSchema()

[df_negocios] Filas: 166751 -> 166751
root
 |-- id: string (nullable = true)
 |-- nom_estab: string (nullable = true)
 |-- raz_social: string (nullable = true)
 |-- codigo_act: string (nullable = true)
 |-- nombre_act: string (nullable = true)
 |-- per_ocu: string (nullable = true)
 |-- tipo_vial: string (nullable = true)
 |-- nom_vial: string (nullable = true)
 |-- tipo_v_e_1: string (nullable = true)
 |-- nom_v_e_1: string (nullable = true)
 |-- tipo_v_e_2: string (nullable = true)
 |-- nom_v_e_2: string (nullable = true)
 |-- tipo_v_e_3: string (nullable = true)
 |-- nom_v_e_3: string (nullable = true)
 |-- numero_ext: string (nullable = true)
 |-- letra_ext: string (nullable = true)
 |-- edificio: string (nullable = true)
 |-- edificio_e: string (nullable = true)
 |-- numero_int: string (nullable = true)
 |-- letra_int: string (nullable = true)
 |-- tipo_asent: string (nullable = true)
 |-- nomb_asent: string (nullable = true)
 |-- tipoCenCom: string (nullable = true)
 |-- nom_CenC

In [0]:
total = df_negocios_clean.count()
cols_criticas = ["codigo_act", "nombre_act", "cve_mun", "cve_loc", "ageb", "manzana", "latitud", "longitud", "fecha_alta"]

df_negocios_clean.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total * 100).alias(c)
    for c in cols_criticas
]).show(vertical=True, truncate=False)

-RECORD 0---------
 codigo_act | 0.0 
 nombre_act | 0.0 
 cve_mun    | 0.0 
 cve_loc    | 0.0 
 ageb       | 0.0 
 manzana    | 0.0 
 latitud    | 0.0 
 longitud   | 0.0 
 fecha_alta | 0.0 



In [0]:
df_restaurantes.printSchema()

root
 |-- id: string (nullable = true)
 |-- nom_estab: string (nullable = true)
 |-- raz_social: string (nullable = true)
 |-- codigo_act: string (nullable = true)
 |-- nombre_act: string (nullable = true)
 |-- per_ocu: string (nullable = true)
 |-- tipo_vial: string (nullable = true)
 |-- nom_vial: string (nullable = true)
 |-- tipo_v_e_1: string (nullable = true)
 |-- nom_v_e_1: string (nullable = true)
 |-- tipo_v_e_2: string (nullable = true)
 |-- nom_v_e_2: string (nullable = true)
 |-- tipo_v_e_3: string (nullable = true)
 |-- nom_v_e_3: string (nullable = true)
 |-- numero_ext: string (nullable = true)
 |-- letra_ext: string (nullable = true)
 |-- edificio: string (nullable = true)
 |-- edificio_e: string (nullable = true)
 |-- numero_int: string (nullable = true)
 |-- letra_int: string (nullable = true)
 |-- tipo_asent: string (nullable = true)
 |-- nomb_asent: string (nullable = true)
 |-- tipoCenCom: string (nullable = true)
 |-- nom_CenCom: string (nullable = true)
 |-- num_

In [0]:
df_restaurantes_clean = limpiar_denue(df_restaurantes, "df_restaurantes")

df_restaurantes_clean.printSchema()

[df_restaurantes] Filas: 185532 -> 185532
root
 |-- id: string (nullable = true)
 |-- nom_estab: string (nullable = true)
 |-- raz_social: string (nullable = true)
 |-- codigo_act: string (nullable = true)
 |-- nombre_act: string (nullable = true)
 |-- per_ocu: string (nullable = true)
 |-- tipo_vial: string (nullable = true)
 |-- nom_vial: string (nullable = true)
 |-- tipo_v_e_1: string (nullable = true)
 |-- nom_v_e_1: string (nullable = true)
 |-- tipo_v_e_2: string (nullable = true)
 |-- nom_v_e_2: string (nullable = true)
 |-- tipo_v_e_3: string (nullable = true)
 |-- nom_v_e_3: string (nullable = true)
 |-- numero_ext: string (nullable = true)
 |-- letra_ext: string (nullable = true)
 |-- edificio: string (nullable = true)
 |-- edificio_e: string (nullable = true)
 |-- numero_int: string (nullable = true)
 |-- letra_int: string (nullable = true)
 |-- tipo_asent: string (nullable = true)
 |-- nomb_asent: string (nullable = true)
 |-- tipoCenCom: string (nullable = true)
 |-- nom_

In [0]:
total = df_restaurantes_clean.count()
cols_criticas = ["codigo_act", "nombre_act", "cve_mun", "cve_loc", "ageb", "manzana", "latitud", "longitud", "fecha_alta"]

df_restaurantes_clean.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total * 100).alias(c)
    for c in cols_criticas
]).show(vertical=True, truncate=False)

-RECORD 0---------
 codigo_act | 0.0 
 nombre_act | 0.0 
 cve_mun    | 0.0 
 cve_loc    | 0.0 
 ageb       | 0.0 
 manzana    | 0.0 
 latitud    | 0.0 
 longitud   | 0.0 
 fecha_alta | 0.0 



In [0]:

df_tiendas.printSchema()

root
 |-- id: string (nullable = true)
 |-- nom_estab: string (nullable = true)
 |-- raz_social: string (nullable = true)
 |-- codigo_act: string (nullable = true)
 |-- nombre_act: string (nullable = true)
 |-- per_ocu: string (nullable = true)
 |-- tipo_vial: string (nullable = true)
 |-- nom_vial: string (nullable = true)
 |-- tipo_v_e_1: string (nullable = true)
 |-- nom_v_e_1: string (nullable = true)
 |-- tipo_v_e_2: string (nullable = true)
 |-- nom_v_e_2: string (nullable = true)
 |-- tipo_v_e_3: string (nullable = true)
 |-- nom_v_e_3: string (nullable = true)
 |-- numero_ext: string (nullable = true)
 |-- letra_ext: string (nullable = true)
 |-- edificio: string (nullable = true)
 |-- edificio_e: string (nullable = true)
 |-- numero_int: string (nullable = true)
 |-- letra_int: string (nullable = true)
 |-- tipo_asent: string (nullable = true)
 |-- nomb_asent: string (nullable = true)
 |-- tipoCenCom: string (nullable = true)
 |-- nom_CenCom: string (nullable = true)
 |-- num_

In [0]:
df_tiendas_clean = limpiar_denue(df_tiendas, "df_tiendas")

total = df_tiendas_clean.count()
cols_criticas = ["codigo_act", "nombre_act", "cve_mun", "cve_loc", "ageb", "manzana", "latitud", "longitud", "fecha_alta"]

df_tiendas_clean.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total * 100).alias(c)
    for c in cols_criticas
]).show(vertical=True, truncate=False)

[df_tiendas] 1 filas sospechosas de corrupción de columnas:
+-------+-------------------------------------------+------------+
|id     |nom_estab                                  |fecha_alta  |
+-------+-------------------------------------------+------------+
|6723590|AZUL CONCRETOS Y PREMEZCLADOS, S.A. DE C.V.|-94.98845153|
+-------+-------------------------------------------+------------+

[df_tiendas] Filas: 608178 -> 608177
-RECORD 0---------
 codigo_act | 0.0 
 nombre_act | 0.0 
 cve_mun    | 0.0 
 cve_loc    | 0.0 
 ageb       | 0.0 
 manzana    | 0.0 
 latitud    | 0.0 
 longitud   | 0.0 
 fecha_alta | 0.0 



In [0]:

df_transporte.printSchema()

root
 |-- id: string (nullable = true)
 |-- nom_estab: string (nullable = true)
 |-- raz_social: string (nullable = true)
 |-- codigo_act: string (nullable = true)
 |-- nombre_act: string (nullable = true)
 |-- per_ocu: string (nullable = true)
 |-- tipo_vial: string (nullable = true)
 |-- nom_vial: string (nullable = true)
 |-- tipo_v_e_1: string (nullable = true)
 |-- nom_v_e_1: string (nullable = true)
 |-- tipo_v_e_2: string (nullable = true)
 |-- nom_v_e_2: string (nullable = true)
 |-- tipo_v_e_3: string (nullable = true)
 |-- nom_v_e_3: string (nullable = true)
 |-- numero_ext: string (nullable = true)
 |-- letra_ext: string (nullable = true)
 |-- edificio: string (nullable = true)
 |-- edificio_e: string (nullable = true)
 |-- numero_int: string (nullable = true)
 |-- letra_int: string (nullable = true)
 |-- tipo_asent: string (nullable = true)
 |-- nomb_asent: string (nullable = true)
 |-- tipoCenCom: string (nullable = true)
 |-- nom_CenCom: string (nullable = true)
 |-- num_

In [0]:
df_transporte_clean = limpiar_denue(df_transporte, "df_transporte")

total = df_transporte_clean.count()
cols_criticas = ["codigo_act", "nombre_act", "cve_mun", "cve_loc", "ageb", "manzana", "latitud", "longitud", "fecha_alta"]

df_transporte_clean.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total * 100).alias(c)
    for c in cols_criticas
]).show(vertical=True, truncate=False)

[df_transporte] Filas: 39021 -> 39021
-RECORD 0---------
 codigo_act | 0.0 
 nombre_act | 0.0 
 cve_mun    | 0.0 
 cve_loc    | 0.0 
 ageb       | 0.0 
 manzana    | 0.0 
 latitud    | 0.0 
 longitud   | 0.0 
 fecha_alta | 0.0 



In [0]:
df_negocios_tiendas.printSchema()

root
 |-- id: string (nullable = true)
 |-- nom_estab: string (nullable = true)
 |-- raz_social: string (nullable = true)
 |-- codigo_act: string (nullable = true)
 |-- nombre_act: string (nullable = true)
 |-- per_ocu: string (nullable = true)
 |-- tipo_vial: string (nullable = true)
 |-- nom_vial: string (nullable = true)
 |-- tipo_v_e_1: string (nullable = true)
 |-- nom_v_e_1: string (nullable = true)
 |-- tipo_v_e_2: string (nullable = true)
 |-- nom_v_e_2: string (nullable = true)
 |-- tipo_v_e_3: string (nullable = true)
 |-- nom_v_e_3: string (nullable = true)
 |-- numero_ext: string (nullable = true)
 |-- letra_ext: string (nullable = true)
 |-- edificio: string (nullable = true)
 |-- edificio_e: string (nullable = true)
 |-- numero_int: string (nullable = true)
 |-- letra_int: string (nullable = true)
 |-- tipo_asent: string (nullable = true)
 |-- nomb_asent: string (nullable = true)
 |-- tipoCenCom: string (nullable = true)
 |-- nom_CenCom: string (nullable = true)
 |-- num_

In [0]:
df_negocios_tiendas_clean = limpiar_denue(df_negocios_tiendas, "df_negocios_tiendas")

total = df_negocios_tiendas_clean.count()
cols_criticas = ["codigo_act", "nombre_act", "cve_mun", "cve_loc", "ageb", "manzana", "latitud", "longitud", "fecha_alta"]

df_negocios_tiendas_clean.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total * 100).alias(c)
    for c in cols_criticas
]).show(vertical=True, truncate=False)

[df_negocios_tiendas] Filas: 73002 -> 73002
-RECORD 0---------
 codigo_act | 0.0 
 nombre_act | 0.0 
 cve_mun    | 0.0 
 cve_loc    | 0.0 
 ageb       | 0.0 
 manzana    | 0.0 
 latitud    | 0.0 
 longitud   | 0.0 
 fecha_alta | 0.0 

